# 01 - Reading and Exploring the Olist Dataset

## 1. Configuration

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Navigate up through the directories until you find `requirements.txt` (in the project root)."""
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists():
            return p
    raise FileNotFoundError("Projet root not found (requirements.txt)")


ROOT = find_project_root()
RAW = ROOT / "data" / "raw" / "olist"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Racine :", ROOT)

Racine : c:\Users\DELL\Downloads\projet_ai


## 2. Vérification des fichiers


In [3]:
FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}


## 3. Chargement


In [13]:
ZIP_COLS = ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]

dfs = {}
for name, fname in FILES.items():
    header = pd.read_csv(RAW / fname, nrows=0).columns
    dtype = {c: "string" for c in header if c in ZIP_COLS}
    dfs[name] = pd.read_csv(RAW / fname, dtype=dtype)

summary = pd.DataFrame(
    {
        "lines": {k: len(v) for k, v in dfs.items()},
        "columns": {k: v.shape[1] for k, v in dfs.items()},
    }
)
summary

,lines,columns
customers,99441,5
orders,99441,8
items,112650,7
payments,103886,5
reviews,99224,7
products,32951,9
sellers,3095,4
geolocation,1000163,5
category_translation,71,2


## 4. Aperçu de chaque table

In [6]:
for name, df in dfs.items():
    print(f"\n===== {name} {df.shape} =====")
    display(df.head(3))


===== customers (99441, 5) =====


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP



===== orders (99441, 8) =====


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



===== items (112650, 7) =====


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


... (8 outputs left out)


===== geolocation (1000163, 5) =====


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP



===== category_translation (71, 2) =====


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 5. Types et valeurs manquantes

In [7]:
rows = []
for name, df in dfs.items():
    for col in df.columns:
        n_null = int(df[col].isna().sum())
        rows.append(
            {
                "table": name,
                "column": col,
                "type": str(df[col].dtype),
                "missing": n_null,
                "% missing": round(100 * n_null / len(df), 2),
                "unique values": df[col].nunique(),
            }
        )
profile = pd.DataFrame(rows)
display(profile[profile["missing"] > 0].sort_values("% missing", ascending=False))

,table,column,type,missing,% missing,unique values
28,reviews,review_comment_title,str,87656,88.34,4527
29,reviews,review_comment_message,str,58247,58.70,36159
11,orders,order_delivered_customer_date,str,2965,2.98,95664
34,products,product_name_lenght,float64,610,1.85,66
33,products,product_category_name,str,610,1.85,73
35,products,product_description_lenght,float64,610,1.85,2960
36,products,product_photos_qty,float64,610,1.85,19
10,orders,order_delivered_carrier_date,str,1783,1.79,81018
9,orders,order_approved_at,str,160,0.16,90733
37,products,product_weight_g,float64,2,0.01,2204


## 6. Doublons

On compte les lignes entièrement dupliquées, puis les doublons sur la clé attendue de chaque table.

In [8]:
KEYS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": None,  # not a key: a postal code appears several times
    "category_translation": ["product_category_name"],
}

rows = []
for name, df in dfs.items():
    key = KEYS[name]
    rows.append(
        {
            "table": name,
            "lines duplicated": int(df.duplicated().sum()),
            "key": ", ".join(key) if key else "-",
            "duplicates on key": int(df.duplicated(subset=key).sum()) if key else None,
        }
    )
pd.DataFrame(rows)

,table,lines duplicated,key,duplicates on key
0,customers,0,customer_id,0.0
1,orders,0,order_id,0.0
2,items,0,"order_id, order_item_id",0.0
3,payments,0,"order_id, payment_sequential",0.0
4,reviews,0,"review_id, order_id",0.0
5,products,0,product_id,0.0
6,sellers,0,seller_id,0.0
7,geolocation,261831,-,NaN
8,category_translation,0,product_category_name,0.0


## 7. Plages de dates et statuts de commande

In [14]:
orders = dfs["orders"]
date_cols = [c for c in orders.columns if c.endswith("_at") or c.endswith("_date")]
dates = orders[date_cols].apply(pd.to_datetime, errors="coerce")
display(pd.DataFrame({"min": dates.min(), "max": dates.max(), "manquantes": dates.isna().sum()}))

print("Statuts de commande :")
display(orders["order_status"].value_counts().to_frame("commandes"))

,min,max,manquantes
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


Statuts de commande :


,commandes
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## 8. Intégrité référentielle

Chaque identifiant d'une table « enfant » doit exister dans la table « parent ».

In [15]:
def orphans(child, child_col, parent, parent_col):
    c = dfs[child][child_col].dropna()
    return int((~c.isin(dfs[parent][parent_col])).sum())


checks = [
    ("orders.customer_id -> customers", orphans("orders", "customer_id", "customers", "customer_id")),
    ("items.order_id -> orders", orphans("items", "order_id", "orders", "order_id")),
    ("items.product_id -> products", orphans("items", "product_id", "products", "product_id")),
    ("items.seller_id -> sellers", orphans("items", "seller_id", "sellers", "seller_id")),
    ("payments.order_id -> orders", orphans("payments", "order_id", "orders", "order_id")),
    ("reviews.order_id -> orders", orphans("reviews", "order_id", "orders", "order_id")),
]
pd.DataFrame(checks, columns=["relation", "lignes orphelines"])

,relation,lignes orphelines
0,orders.customer_id -> customers,0
1,items.order_id -> orders,0
2,items.product_id -> products,0
3,items.seller_id -> sellers,0
4,payments.order_id -> orders,0
5,reviews.order_id -> orders,0


## 9. Statistiques rapides sur les ventes (items)

In [16]:
items = dfs["items"]
display(items[["price", "freight_value"]].describe().round(2))
print("Produits distincts vendus :", items["product_id"].nunique())
sales_per_product = items.groupby("product_id").size()
print("Produits avec moins de 5 ventes :", f"{(sales_per_product < 5).mean():.1%}")
print("Produits avec au moins 50 ventes :", int((sales_per_product >= 50).sum()))

,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68


Produits distincts vendus : 32951
Produits avec moins de 5 ventes : 85.3%
Produits avec au moins 50 ventes : 174


## 10. Constats à reporter dans le notebook de nettoyage

À compléter après exécution :

- Colonnes avec valeurs manquantes importantes : ...
- Doublons trouvés : ...
- Lignes orphelines : ...
- Autres anomalies : ...